# Pathological Gait Recognition: Three-Experiment Comparison

This notebook compares **three experimental approaches** for pathological gait recognition using VAE and Contrastive VAE models:

## Experiments:
1. **Experiment 1 (Zero-Shot Transfer)**: Models trained on CASIA-B (healthy gait), tested on pathology dataset
   - VAE: ✅ Available (`exp1_vae_casia.pth`)
   - Contrastive VAE: 🔄 Training on CASIA-B data...
   
2. **Experiment 2 (Fine-Tuning)**: Models trained on CASIA-B, then fine-tuned on pathology data
   - VAE Fine-tuned: ⏳ Will fine-tune after Exp1 complete
   - Contrastive VAE Fine-tuned: ⏳ Will fine-tune after Exp1 complete
   
3. **Experiment 3 (From Scratch)**: Models trained entirely on pathology dataset
   - VAE From Scratch: ✅ Available (`exp3_vae_pathology.pth`)
   - Contrastive VAE From Scratch: ✅ Available (`exp3_contrastive_pathology.pth`)

## Data Sources:
- **CASIA-B**: 124 subjects, ~13,500 GEI samples from `preprocessing/preprocessed_data/`
- **Pathology**: 53 subjects, 1,288 images (Normal, Parkinson, Diplegic, Hemiplegic, Neuropathic)

## Goals:
- Compare zero-shot transfer vs fine-tuning vs scratch training
- Evaluate binary classification (healthy vs pathological)
- Evaluate multi-class classification (specific conditions)
- Assess cross-view subject identification
- Determine which training regime produces best embeddings for pathological gait analysis


In [ ]:
from model import create_vae
from contrastive_model import create_contrastive_vae

# Define checkpoint paths for all experiments
CHECKPOINTS = {
    'exp1': {
        'vae': Path('../checkpoints/exp1_vae_casia.pth'),  # CASIA-B trained ✅
        'contrastive': Path('../checkpoints/exp1_contrastive_casia.pth')  # CASIA-B trained (training...)
    },
    'exp2': {
        'vae': Path('../checkpoints/exp2_vae_finetuned.pth'),  # Fine-tuned from CASIA-B
        'contrastive': Path('../checkpoints/exp2_contrastive_finetuned.pth')  # Fine-tuned from CASIA-B
    },
    'exp3': {
        'vae': Path('../checkpoints/exp3_vae_pathology.pth'),  # Pathology trained from scratch ✅
        'contrastive': Path('../checkpoints/exp3_contrastive_pathology.pth')  # Pathology trained from scratch ✅
    }
}

# Load all available models
models = {}
model_info = {}

print("=" * 80)
print("LOADING MODELS FOR ALL EXPERIMENTS")
print("=" * 80)

for exp_name, exp_checkpoints in CHECKPOINTS.items():
    models[exp_name] = {}
    model_info[exp_name] = {}
    
    print(f"\n{exp_name.upper()}")
    print("-" * 80)
    
    # Load VAE
    if exp_checkpoints['vae'] and exp_checkpoints['vae'].exists():
        print(f"  Loading VAE from {exp_checkpoints['vae'].name}...")
        vae = create_vae(latent_dim=128)
        ckpt = torch.load(exp_checkpoints['vae'], map_location=device)
        vae.load_state_dict(ckpt['model_state_dict'])
        vae = vae.to(device)
        vae.eval()
        models[exp_name]['vae'] = vae
        model_info[exp_name]['vae'] = {
            'epoch': ckpt.get('epoch', 'N/A'),
            'loss': ckpt.get('val_total_loss', 'N/A')
        }
        print(f"  ✅ VAE loaded (Epoch: {ckpt.get('epoch', 'N/A')})")
    else:
        print(f"  ⏳ VAE checkpoint not found")
        models[exp_name]['vae'] = None
    
    # Load Contrastive VAE
    if exp_checkpoints['contrastive'] and exp_checkpoints['contrastive'].exists():
        print(f"  Loading Contrastive VAE from {exp_checkpoints['contrastive'].name}...")
        contrastive_vae = create_contrastive_vae(latent_dim=128, projection_dim=128)
        ckpt = torch.load(exp_checkpoints['contrastive'], map_location=device)
        contrastive_vae.load_state_dict(ckpt['model_state_dict'])
        contrastive_vae = contrastive_vae.to(device)
        contrastive_vae.eval()
        models[exp_name]['contrastive'] = contrastive_vae
        model_info[exp_name]['contrastive'] = {
            'epoch': ckpt.get('epoch', 'N/A'),
            'loss': ckpt.get('val_total_loss', 'N/A'),
            'contrastive_loss': ckpt.get('val_contrastive_loss', 'N/A')
        }
        print(f"  ✅ Contrastive VAE loaded (Epoch: {ckpt.get('epoch', 'N/A')})")
    else:
        print(f"  ⏳ Contrastive VAE checkpoint not found")
        models[exp_name]['contrastive'] = None

print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)
for exp_name in models:
    print(f"{exp_name.upper()}: VAE={'✅' if models[exp_name]['vae'] else '⏳'}, Contrastive={'✅' if models[exp_name]['contrastive'] else '⏳'}")


In [ ]:
def load_and_preprocess_image(image_path, target_size=(128, 64)):
    """Load GEI-like image and convert to tensor."""
    img = Image.open(image_path).convert('L')
    img = img.resize(target_size, Image.LANCZOS)
    arr = np.array(img, dtype=np.float32) / 255.0
    return torch.from_numpy(arr).unsqueeze(0).unsqueeze(0)

def collect_pathology_samples(root, gei_patterns=("*.png", "*.jpg")):
    """Collect pathological samples with metadata."""
    samples = {"paths": [], "subjects": [], "conditions": []}
    root = Path(root)
    if not root.exists():
        print(f"⚠️ Pathology data not found at {root}")
        return samples
    
    for cond_dir in sorted(d for d in root.iterdir() if d.is_dir()):
        condition_name = cond_dir.name.lower()
        subject_dirs = sorted(d for d in cond_dir.iterdir() if d.is_dir())
        if not subject_dirs:
            subject_dirs = [cond_dir]
        
        for subj_dir in subject_dirs:
            subject_id = subj_dir.name
            img_files = []
            for pat in gei_patterns:
                img_files.extend(subj_dir.rglob(pat))
            
            for img_path in img_files:
                samples["paths"].append(img_path)
                samples["subjects"].append(subject_id)
                samples["conditions"].append(condition_name)
    
    print(f"📸 Collected {len(samples['paths'])} images")
    print(f"   Subjects: {len(set(samples['subjects']))}")
    print(f"   Conditions: {sorted(set(samples['conditions']))}")
    return samples

# Load pathology dataset
PATHOLOGY_ROOT = Path('../data/pathology_data_for_training')
all_samples = collect_pathology_samples(PATHOLOGY_ROOT)

# Convert to arrays for easier manipulation
all_paths = all_samples['paths']
all_subjects = np.array(all_samples['subjects'])
all_conditions = np.array(all_samples['conditions'])

print(f"\nTotal samples: {len(all_paths)}")
print(f"Condition distribution: {dict(Counter(all_conditions))}")


## 2. Data Loading and Preprocessing

## 1. Load Models for All Experiments

In [ ]:
# Setup and imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random
import seaborn as sns
from pathlib import Path
import sys
import warnings
warnings.filterwarnings('ignore')

# Deep learning
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

# Computer vision
from PIL import Image

# Dimensionality reduction and visualization
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.neighbors import NearestNeighbors
from collections import Counter

# Set seeds for reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Add parent directory to path for imports
sys.path.insert(0, str(Path('..').resolve()))
